# QuantJourney SDK - Corporate Actions and Adjustment Semantics

This notebook demonstrates a QuantJourney SDK workflow that checks adjusted price consistency, dividend history and corporate-action evidence around a single equity.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


In [ ]:
symbol = 'AAPL'
prices_raw = qj.eod.get_historical_prices(symbol=symbol, start_date='2018-01-01', end_date=END)
dividends_raw = qj.fmp.get_dividends_historical(symbol=symbol)
last_dividend_raw = qj.fmp.get_last_dividend(symbol=symbol)
shares_raw = qj.eod.get_shares_stats(symbol=symbol)
profile_raw = qj.fmp.get_company_profile(symbol=symbol)


In [ ]:
prices = pd.DataFrame(as_rows(prices_raw))
if not prices.empty:
    prices['date'] = pd.to_datetime(prices.get('date'), errors='coerce')
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in prices:
            prices[col] = pd.to_numeric(prices[col], errors='coerce')
    prices = prices.dropna(subset=['date']).sort_values('date').set_index('date')
    if 'adjusted_close' in prices and 'close' in prices:
        prices['adjustment_ratio'] = prices['adjusted_close'] / prices['close']
        prices['adjustment_gap'] = prices['adjusted_close'] - prices['close']
dividends = pd.DataFrame(as_rows(dividends_raw))
if not dividends.empty:
    date_col = next((col for col in dividends.columns if 'date' in str(col).lower()), dividends.columns[0])
    value_col = next((col for col in dividends.columns if 'dividend' in str(col).lower() or 'adj' in str(col).lower()), None)
    dividends['date'] = pd.to_datetime(dividends[date_col], errors='coerce')
    if value_col:
        dividends['dividend'] = pd.to_numeric(dividends[value_col], errors='coerce')
    dividends = dividends.dropna(subset=['date']).sort_values('date')


In [ ]:
audit = pd.Series({'price_rows': len(prices), 'dividend_rows': len(dividends), 'last_dividend_available': bool(as_rows(last_dividend_raw)), 'shares_stats_available': bool(as_rows(shares_raw)), 'profile_available': bool(as_rows(profile_raw)), 'has_adjusted_close': 'adjusted_close' in prices.columns if not prices.empty else False, 'adjustment_changes': int(prices['adjustment_ratio'].diff().abs().gt(0.001).sum()) if 'adjustment_ratio' in prices else 0})
display(audit)
if not prices.empty and {'close', 'adjusted_close'}.issubset(prices.columns):
    prices[['close', 'adjusted_close']].dropna().tail(1000).plot(title='Close vs adjusted close')
    plt.ylabel('price')
    plt.show()
if not dividends.empty and 'dividend' in dividends:
    dividends.tail(40).set_index('date')['dividend'].plot(kind='bar', title='Recent dividend events')
    plt.ylabel('cash dividend')
    plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.